In [96]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import os
import shutil

In [97]:
input_data_path = r'C:\Users\songy\Downloads\UVA STAT 5630 Statistical Machine Learning\Project\data\chest_xray_resized'
output_data_path = r'C:\Users\songy\Downloads\UVA STAT 5630 Statistical Machine Learning\Project\data\chest_xray_resized_new'

In [98]:
file_paths = pd.read_csv("data/chest_xray_resized/file_paths.csv")

In [99]:
## random training/validation splitting (80/20):
## make sure that if a patient has multiple X-Ray images, then all images belonging to the patient either enter the training or the validation set
np.random.seed(42)
shuffled_patient_id = pd.DataFrame({"patient_id":np.random.permutation(np.unique(file_paths[file_paths.dataset=="train"]["patient_id"]))})
shuffled_patient_id = shuffled_patient_id.merge(file_paths.groupby(['patient_id']).agg({'image_path':'count'}).reset_index(), on=['patient_id'], how="inner")
train_patient_id = np.array(shuffled_patient_id[np.cumsum(shuffled_patient_id['image_path']) <= 0.8*Counter(file_paths.dataset)['train']].patient_id)
val_patient_id = np.setdiff1d(shuffled_patient_id.patient_id, train_patient_id)
test_patient_id = np.unique(file_paths[file_paths.dataset == "test"].patient_id)
dataset_new = pd.DataFrame({"patient_id": list(train_patient_id) + list(val_patient_id) + list(test_patient_id),
                            "dataset_new": ['train']*len(train_patient_id) + ['val']*len(val_patient_id) + ['test']*len(test_patient_id)})
file_paths = file_paths.merge(dataset_new, on=['patient_id'], how='inner')

In [100]:
try:
    os.makedirs(output_data_path, exist_ok=False)
except FileExistsError:
    pass

for dataset in ['train', 'val', 'test']:
    try:
        os.makedirs(os.path.join(output_data_path, dataset), exist_ok=False)
    except FileExistsError:
        pass
    for class_name in ['NORMAL', 'PNEUMONIA']:
        try:
            os.makedirs(os.path.join(output_data_path, dataset, class_name), exist_ok=False)
        except FileExistsError:
            pass

In [103]:
image_path_new_list = []
for j in range(file_paths.shape[0]):
    image_path = file_paths.image_path.iloc[j]
    dataset = file_paths.dataset_new.iloc[j]
    image_path_old = input_data_path + "/" + "/".join(image_path.split("/")[1:])
    image_path_new = output_data_path + "/" + dataset + "/" + "/".join(image_path.split("/")[2:])
    image_path_new_list.append(output_data_path.split("\\")[-1] + "/" + dataset + "/" + "/".join(image_path.split("/")[2:]))
    shutil.copy(image_path_old, image_path_new)

In [107]:
# file_paths['image_path'] = image_path_new_list
# file_paths['dataset'] = file_paths.dataset_new
# file_paths = file_paths.drop(['dataset_new'], axis=1)
file_paths.to_csv(output_data_path + "/file_paths.csv", index=False)